# 05 · ALS và vector phiên (ticket #14)

**Input** (Add Input): Kaggle Dataset `titeohihi/mixnext-splited` (output của notebook 03).

**Output** (`/kaggle/working/out/`):
- `candidates/als_{val,test}_run0.parquet`
- `item_embeddings/als.npy`, `item_embeddings/als_tracks.parquet`
- `als_params.json`: tham số đã chọn và NDCG@10 trên val của cả lưới (lần chạy 1, 2 ở #26 dùng lại file này qua `--params`)

Lưới chọn trên val (lần 2): factors {128, 256} × regularization {0,1; 1; 10} × decay {0,5; 0,7; 0,9; 1,0} (24 cấu hình, 6 lần học ALS). Chọn theo NDCG@10. Lần 1 dùng lưới nhỏ hơn và tốt nhất nằm ở mép lưới, xem mục Kết quả.

Bật **Internet** (clone repo, `pip install implicit`). Chỉ cần CPU. Ước tính 35–45 phút.

In [ ]:
# Bootstrap: lấy code từ GitHub. Khi #14 đã merge vào main thì đổi BRANCH = "main".
BRANCH = "b"
!rm -rf /kaggle/working/MusicRecSys && git clone -q --depth 1 -b {BRANCH} https://github.com/yashita23/MusicRecSys /kaggle/working/MusicRecSys
import sys; sys.path.insert(0, "/kaggle/working/MusicRecSys")

In [ ]:
# implicit không có sẵn trên Kaggle (#2)
!pip install -q implicit==0.7.3

In [ ]:
from pathlib import Path
DATA = sorted(Path("/kaggle/input").rglob("eval_targets.parquet"))[0].parent
OUT = Path("/kaggle/working/out")
print("input:", DATA, sorted(p.name for p in DATA.iterdir()))

In [ ]:
import time
from src.models.als import main

t0 = time.time()
report = main(["--data", str(DATA), "--out", str(OUT)])   # chọn tham số trên val, xuất file, có validate_dataset bên trong
print(f"{time.time() - t0:.0f} giây")
for p in sorted(OUT.rglob("*.*")):
    print(p.relative_to(OUT), f"{p.stat().st_size / 1e6:.1f} MB")

In [ ]:
# Lưới trên val, xếp theo NDCG@10, so với hai baseline của #9 (notebook 04, val 9.212 phiên)
import pandas as pd

BASELINE_NDCG = {"popularity": 0.0040, "itemknn": 0.1297}
grid = pd.DataFrame(report["grid"]).sort_values("ndcg", ascending=False)
print(grid.round(4).to_string(index=False))
ndcg = report["val"]["ndcg"]
print(f"\nChọn: {report['params']} | NDCG@10 val {ndcg:.4f}")
for name, base in BASELINE_NDCG.items():
    print(f"  so với {name} {base}: {ndcg / base - 1:+.1%}")

## Đăng output

1. **Save Version → Save & Run All (Commit)** để lưu output.
2. Theo `docs/data-contract.md`, output chia về hai Dataset:
   - Tab Output của version vừa chạy → **New Dataset** tên `mixnext-embeddings` (gồm `item_embeddings/als.*` và `als_params.json`). Embedding này cho app (#20) và sinh chuỗi (#18); item2vec (#15) thêm vào version sau.
   - Tải hai file `candidates/als_{val,test}_run0.parquet` ở tab Output, rồi mở Dataset `mixnext-candidates` → **New Version** → upload vào thư mục `candidates/`, giữ các file của Popularity và item-kNN.

## Kết quả trên val

### Lần 1 (Kaggle, CPU, 26/11/2026, 896 giây): factors {64, 128} × regularization {0,01; 0,1} × decay {0,5; 0,7; 0,9; 1,0}

| factors | regularization | decay | HitRate@10 | NDCG@10 | MRR |
|---:|---:|---:|---:|---:|---:|
| 128 | 0,1 | 0,5 | 0,1021 | **0,0542** | 0,0471 |
| 128 | 0,1 | 0,7 | 0,0998 | 0,0519 | 0,0447 |
| 128 | 0,1 | 0,9 | 0,0943 | 0,0491 | 0,0426 |
| 128 | 0,1 | 1,0 | 0,0928 | 0,0482 | 0,0417 |
| 128 | 0,01 | 0,5 | 0,0835 | 0,0440 | 0,0386 |
| 128 | 0,01 | 0,7 | 0,0814 | 0,0421 | 0,0365 |
| 128 | 0,01 | 0,9 | 0,0779 | 0,0402 | 0,0351 |
| 128 | 0,01 | 1,0 | 0,0759 | 0,0391 | 0,0342 |
| 64 | 0,1 | 0,5 | 0,0464 | 0,0223 | 0,0196 |
| 64 | 0,1 | 0,7 | 0,0452 | 0,0216 | 0,0189 |
| 64 | 0,1 | 0,9 | 0,0437 | 0,0210 | 0,0185 |
| 64 | 0,1 | 1,0 | 0,0426 | 0,0207 | 0,0184 |
| 64 | 0,01 | 0,5 | 0,0330 | 0,0150 | 0,0133 |
| 64 | 0,01 | 0,7 | 0,0320 | 0,0146 | 0,0131 |
| 64 | 0,01 | 1,0 | 0,0319 | 0,0144 | 0,0127 |
| 64 | 0,01 | 0,9 | 0,0318 | 0,0144 | 0,0127 |

Chọn (128; 0,1; 0,5): NDCG@10 val 0,0542, hơn Popularity (0,0040) 13,6 lần nhưng kém item-kNN liền kề (0,1297) 58%.

- Cả ba tham số tốt nhất nằm ở mép lưới: factors 64 → 128 tăng NDCG@10 khoảng 2,4 lần (danh mục khoảng 130 nghìn bài), regularization 0,1 hơn 0,01 ở mọi dòng. Vì vậy lần 2 mở rộng về phía factors và regularization lớn hơn.
- decay 0,5 tốt nhất ở cả bốn cặp: bài gần cuối phiên càng nặng càng tốt, cùng chiều với việc item-kNN liền kề (chỉ nhìn bài cuối) mạnh. Tập decay giữ đúng spec §6 {0,5; 0,7; 0,9; 1,0}.

### Lần 2 (Kaggle, CPU, 26/11/2026, 2.099 giây): factors {128, 256} × regularization {0,1; 1; 10} × decay {0,5; 0,7; 0,9; 1,0}

| factors | regularization | decay | HitRate@10 | NDCG@10 | MRR |
|---:|---:|---:|---:|---:|---:|
| 256 | 1 | 0,5 | 0,1709 | **0,0967** | 0,0829 |
| 256 | 1 | 0,7 | 0,1702 | 0,0954 | 0,0818 |
| 256 | 10 | 0,5 | 0,1689 | 0,0950 | 0,0815 |
| 256 | 10 | 0,7 | 0,1659 | 0,0930 | 0,0802 |
| 256 | 1 | 0,9 | 0,1624 | 0,0886 | 0,0755 |
| 256 | 0,1 | 0,5 | 0,1582 | 0,0884 | 0,0758 |
| 256 | 10 | 0,9 | 0,1617 | 0,0879 | 0,0747 |
| 256 | 0,1 | 0,7 | 0,1551 | 0,0866 | 0,0746 |
| 256 | 1 | 1,0 | 0,1574 | 0,0850 | 0,0723 |
| 256 | 10 | 1,0 | 0,1553 | 0,0843 | 0,0720 |
| 256 | 0,1 | 0,9 | 0,1481 | 0,0812 | 0,0701 |
| 128 | 10 | 0,5 | 0,1442 | 0,0809 | 0,0706 |
| 128 | 10 | 0,7 | 0,1469 | 0,0806 | 0,0694 |
| 128 | 1 | 0,5 | 0,1432 | 0,0793 | 0,0689 |
| 128 | 1 | 0,7 | 0,1446 | 0,0781 | 0,0668 |
| 256 | 0,1 | 1,0 | 0,1436 | 0,0779 | 0,0670 |
| 128 | 10 | 0,9 | 0,1408 | 0,0766 | 0,0662 |
| 128 | 10 | 1,0 | 0,1354 | 0,0736 | 0,0639 |
| 128 | 1 | 0,9 | 0,1334 | 0,0727 | 0,0635 |
| 128 | 1 | 1,0 | 0,1304 | 0,0705 | 0,0614 |
| 128 | 0,1 | 0,5 | 0,1021 | 0,0542 | 0,0471 |
| 128 | 0,1 | 0,7 | 0,0998 | 0,0519 | 0,0447 |
| 128 | 0,1 | 0,9 | 0,0943 | 0,0491 | 0,0426 |
| 128 | 0,1 | 1,0 | 0,0928 | 0,0482 | 0,0417 |

**Chốt: factors 256, regularization 1, decay 0,5** (`als_params.json`). NDCG@10 val 0,0967, hơn Popularity (0,0040) 24 lần, kém item-kNN liền kề (0,1297) 25,5%.

- regularization 1 nằm giữa lưới: hơn cả 0,1 và 10 ở factors 256.
- factors 256 vẫn ở mép lưới nhưng lợi ích giảm dần: 64 → 128 tăng NDCG@10 khoảng 2,4 lần, 128 → 256 chỉ khoảng 1,2 lần (0,0809 → 0,0967). Dừng ở 256 vì `als.npy` đã 134 MB, trong khi `artifacts/` của app phải dưới 200 MB (#20); 512 chiều sẽ khoảng 268 MB.
- decay 0,5 tốt nhất ở mọi cặp (factors, regularization), giống lần 1.
- ALS vẫn kém item-kNN liền kề: điểm gu của ALS dựa trên đồng xuất hiện trong cả playlist, còn bài kế tiếp phụ thuộc mạnh vào bài đang phát. item2vec (#15, cửa sổ ngữ cảnh ngắn) và SASRec (#16, theo thứ tự) có thể nắm được điều này tốt hơn.